# 05 — Generate TPC-H Benchmark Data
Run this once before the benchmark notebook.

In [ ]:
import os, time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

GLUTEN_ENABLED = os.environ.get('GLUTEN_ENABLED', 'false').lower() == 'true'
MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data'

spark = (
    SparkSession.builder
    .appName('spark-notebook')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
mode = 'Gluten/Velox' if GLUTEN_ENABLED else 'Vanilla'
print(f'Spark {spark.version}  |  Mode: {mode}')
spark.range(3).show()


In [ ]:
import random, math
from pathlib import Path

SCALE = 1          # 1 = ~150 MB, 5 = ~750 MB
N_LINEITEM  = 600_000 * SCALE
N_ORDERS    = 150_000 * SCALE
N_CUSTOMERS = 15_000  * SCALE

Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
print(f"Generating scale={SCALE}: {N_LINEITEM} lineitems, {N_ORDERS} orders, {N_CUSTOMERS} customers")


In [ ]:
from pyspark.sql.types import *
import datetime

lineitem_schema = StructType([
    StructField('l_orderkey',      LongType()),
    StructField('l_partkey',       LongType()),
    StructField('l_quantity',      DoubleType()),
    StructField('l_extendedprice', DoubleType()),
    StructField('l_discount',      DoubleType()),
    StructField('l_tax',           DoubleType()),
    StructField('l_returnflag',    StringType()),
    StructField('l_linestatus',    StringType()),
    StructField('l_shipdate',      DateType()),
    StructField('l_commitdate',    DateType()),
])

rng = random.Random(42)
base = datetime.date(1992, 1, 1)

def make_lineitem(n):
    rows = []
    for i in range(n):
        qty   = round(rng.uniform(1, 50), 2)
        price = round(qty * rng.uniform(10, 200), 2)
        days  = rng.randint(0, 2557)
        rows.append((
            rng.randint(1, N_ORDERS),
            rng.randint(1, 200000),
            qty, price,
            round(float(rng.choice([0,0.02,0.04,0.06,0.08,0.1])), 2),
            round(rng.uniform(0, 0.08), 4),
            rng.choice(['A','N','R']),
            rng.choice(['F','O']),
            base + datetime.timedelta(days=days),
            base + datetime.timedelta(days=days + rng.randint(10,30)),
        ))
    return rows

li_data = make_lineitem(N_LINEITEM)
lineitem = spark.createDataFrame(li_data, lineitem_schema)
lineitem.write.mode('overwrite').parquet(f'{DATA_DIR}/lineitem.parquet')
print(f'lineitem written: {lineitem.count():,} rows')


In [ ]:
orders_schema = StructType([
    StructField('o_orderkey',    LongType()),
    StructField('o_custkey',     LongType()),
    StructField('o_orderstatus', StringType()),
    StructField('o_totalprice',  DoubleType()),
    StructField('o_orderdate',   DateType()),
    StructField('o_orderpriority', StringType()),
])

o_rows = [(i+1, rng.randint(1,N_CUSTOMERS), rng.choice(['F','O','P']),
           round(rng.uniform(1000,500000),2),
           base+datetime.timedelta(days=rng.randint(0,2557)),
           rng.choice(['1-URGENT','2-HIGH','3-MEDIUM','4-NOT SPECIFIED','5-LOW']))
          for i in range(N_ORDERS)]
orders = spark.createDataFrame(o_rows, orders_schema)
orders.write.mode('overwrite').parquet(f'{DATA_DIR}/orders.parquet')

customer_schema = StructType([
    StructField('c_custkey',  LongType()),
    StructField('c_name',     StringType()),
    StructField('c_nation',   StringType()),
    StructField('c_acctbal',  DoubleType()),
])
nations = ['FRANCE','GERMANY','UNITED STATES','UNITED KINGDOM','JAPAN','BRAZIL','CANADA']
c_rows  = [(i+1, f'Customer#{i+1:06d}', rng.choice(nations), round(rng.uniform(-1000,10000),2))
           for i in range(N_CUSTOMERS)]
customers = spark.createDataFrame(c_rows, customer_schema)
customers.write.mode('overwrite').parquet(f'{DATA_DIR}/customer.parquet')
print(f'orders: {orders.count():,}   customers: {customers.count():,}')
print('Data generation complete.')
